# FinTrace 360: IEEE-CIS Fraud Detection Analytics
**Module:** High-Dimensional E-Commerce Payment Fraud & Memory-Safe Processing  
**Reference:** IEEE Computational Intelligence Society Fraud Detection Benchmark  
**Objective:** Stream large transaction tables via chunking, downcast dtypes, and analyze card/email domain risk features.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from src.ingestion.ieee_cis_loader import load_ieee_cis_chunks

sns.set_theme(style="whitegrid", palette="muted")


## 1. Stream IEEE-CIS in Low-Memory Chunks


In [ ]:
chunks = []
for chunk in load_ieee_cis_chunks(chunksize=10000):
    chunks.append(chunk)
    if len(chunks) >= 3:
        break

df_ieee = pd.concat(chunks, ignore_index=True)
print(f"Loaded Chunk Sample Shape: {df_ieee.shape}")
print(f"Memory Footprint: {df_ieee.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
df_ieee.head()


## 2. Card Network & Product Code Fraud Rates


In [ ]:
card_fraud = df_ieee.groupby("card4").agg(
    total_tx=("TransactionID", "count"),
    fraud_rate=("isFraud", lambda x: x.mean() * 100)
).reset_index()
display(card_fraud)


## 3. Email Domain Risk Profiling


In [ ]:
email_risk = df_ieee.groupby("P_emaildomain").agg(
    total_tx=("TransactionID", "count"),
    fraud_rate=("isFraud", lambda x: x.mean() * 100)
).reset_index().sort_values(by="fraud_rate", ascending=False)
display(email_risk)
